# 🧠 AI Logic Engine: High-Precision Knowledge Ingestion (Firestore)

ဤ Notebook သည် Firestore အတွင်းသို့ Knowledge Triplets (Subject-Verb-Object) များကို **Symbolic Logic** စနစ်ဖြင့် စနစ်တကျ Ingest လုပ်ပေးမည်ဖြစ်သည်။

### 🚀 Optimization & Resilience:
- 🔄 **Resume Support:** `ingestion_state.json` ကြောင့် Session ပြတ်သော်လည်း ရောက်သည့်နေရာမှ ပြန်စနိုင်ပါသည်။
- ⚡ **Batch Performance:** Firestore Batched Writes (500 limit) ကို အသုံးပြု၍ Ingestion Rate ကို မြှင့်ထားပါသည်။
- 🛡️ **Memory Guard:** Long-run များတွင် GPU Memory မပြည့်အောင် Auto-cleanup လုပ်ဆောင်ပါသည်။
- 🔗 **Universal ID Mapper:** Subject နှင့် Object များကို ID အဖြစ်ပြောင်းလဲရာတွင် Inference Engine နှင့် တစ်ထပ်တည်းဖြစ်အောင် Standardized လုပ်ထားပါသည်။

In [ ]:
# @title ⚡ Step 0: Keep-Alive Script
import IPython
js_code = """
function ClickConnect(){
  console.log("Keeping Session Active...");
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.click();
}
setInterval(ClickConnect, 60000)
"""
display(IPython.display.Javascript(js_code))
print("✅ Keep-Alive script running.")

In [ ]:
# @title 🛠️ Step 1: Install & Setup
!pip install -q firebase-admin transformers accelerate bitsandbytes torch tqdm

import os, json, re, torch, time, hashlib
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer

print("✅ Environment Ready.")

In [ ]:
# @title 🔑 Step 2: Firestore Memory Initialization
# @markdown Please ensure you have your `serviceAccountKey.json` from the Firebase Console.
DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Please upload your serviceAccountKey.json file:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Connected to Firestore: {DATABASE_ID}")

In [ ]:
# @title 🧬 Step 3: Symbolic Grammar Engine Core
CHECKPOINT_FILE = "ingestion_state.json"

class GrammarTree:
    def __init__(self, s, v, o, sentence=None):
        # Clean and Normalize data strictly
        self.subject = s.strip().title()
        self.verb = v.strip().lower()
        self.object = o.strip().title()
        # The 'sid' MUST match exactly between Ingestion and Inference
        self.sid = self.clean_id(self.subject)
        self.oid = self.clean_id(self.object)
        self.sentence = sentence.strip() if sentence else f"{self.subject} {self.verb} {self.object}."

    def clean_id(self, text):
        # Normalize to lowercase alphanumeric with underscores
        return re.sub(r'[^a-z0-9]', '_', text.lower().strip())

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f: return json.load(f)
        except: pass
    return {"total_facts": 0, "last_cat_idx": 0}

def save_checkpoint(count, idx):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"total_facts": count, "last_cat_idx": idx}, f)

print("✅ Symbolic Engine Configured.")

In [ ]:
# @title 🤖 Step 4: Load Knowledge Extractor Model
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"⏳ Loading Optimization Model ({model_id})...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype="auto", 
    device_map="auto",
    load_in_4bit=True # Optimized for Colab T4 RAM
)
model.eval()
print("✅ Extraction Engine Ready.")

In [ ]:
# @title 🚀 Step 5: Start High-Volume continuous Ingestion
TARGET_GOAL = 3000000000 # @param {type:"number"}
CATEGORIES = [
    "Fundamental Biology", "Quantum Mechanics", "World History", "Advanced Economics", 
    "Neuroscience", "Computer Science", "Civil Law", "Environmental Science", "Robotics"
]

state = load_checkpoint()
count, cat_idx = state['total_facts'], state['last_cat_idx']
pbar = tqdm(initial=count, total=TARGET_GOAL, desc="Symbolic Extraction")

while count < TARGET_GOAL:
    try:
        loop_start = time.time()
        current_cat = CATEGORIES[cat_idx % len(CATEGORIES)]
        
        prompt = f"Extract 15 diverse, factual symbolic triplets about {current_cat}. Format: Subject|Verb|Object|DescriptiveSentence. No preamble."
        messages = [
            {"role": "system", "content": "You are a symbolic logic generator. Output ONLY data rows separated by '|'."},
            {"role": "user", "content": prompt}
        ]
        
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            out = model.generate(inputs.input_ids, max_new_tokens=800, do_sample=True, temperature=0.8)
        
        response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        batch = db.batch()
        added_this_round = 0
        
        for line in response.strip().split('\n'):
            parts = line.split('|')
            if len(parts) >= 3:
                tree = GrammarTree(parts[0], parts[1], parts[2], parts[3] if len(parts)>3 else None)
                if not tree.sid or not tree.oid: continue
                
                node_ref = db.collection('nodes').document(tree.sid)
                payload = {'name': tree.subject, 'updatedAt': firestore.SERVER_TIMESTAMP}
                
                if tree.verb == 'is_a':
                    payload['groups'] = firestore.ArrayUnion([tree.oid])
                else:
                    payload['relations'] = firestore.ArrayUnion([{
                        'verb': tree.verb, 'targetId': tree.oid, 
                        'sentence': tree.sentence, 'weight': 100
                    }])
                
                batch.set(node_ref, payload, merge=True)
                added_this_round += 1
        
        if added_this_round > 0:
            batch.commit()
            count += added_this_round
            pbar.update(added_this_round)
            
            # Metric display
            rate = added_this_round / (time.time() - loop_start)
            pbar.set_postfix({"rate": f"{rate:.1f} s/s", "cat": current_cat[:10]})
            
        cat_idx += 1
        save_checkpoint(count, cat_idx)
        
        # Memory Maintenance
        if cat_idx % 10 == 0: torch.cuda.empty_cache()
        
    except KeyboardInterrupt: break
    except Exception as e:
        print(f"\n⚠️ Engine Backoff: {e}. Cooling for 10s...")
        time.sleep(10)

pbar.close()
print(f"✅ Cycle Finished. Total Factual Nodes in Memory: {count}")